# 02 - Feature Engineering

**Trabalho Final MBA BI Master - PUC-Rio**

Objetivo: transformar a base tratada no notebook `01` em uma tabela de modelagem,
com uma linha por loja/dia e todas as variaveis explicativas prontas para os
modelos XGBoost e Random Forest.

O notebook `01` ja entregou boa parte do trabalho: merge com o cadastro das lojas,
filtro de dias fechados e varias variaveis de calendario e de concorrencia.
Este notebook complementa o que falta e monta a tabela final.

Blocos de variaveis:

1. Temporais (completa o que faltar do calendario)
2. Cadastro da loja (escala log da distancia do concorrente)
3. Medias historicas por loja
4. Macroeconomicas (series mensais do FRED)
5. Codificacao das categoricas e tabela final
6. Conferencias finais e baseline de referencia

As decisoes de projeto do bloco 3 estao documentadas no arquivo
`decisoes_bloco3_historico_vendas.md`.

## 0. Setup e carga da base tratada

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

CAMINHO = '/content/drive/MyDrive/Trabalho MBA/'

df = pd.read_parquet(CAMINHO + 'dados_tratados.parquet')

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

print('Linhas e colunas:', df.shape)
print('Periodo:', df['Date'].min().date(), 'a', df['Date'].max().date())
print('Lojas:', df['Store'].nunique())
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Linhas e colunas: (844338, 22)
Periodo: 2013-01-01 a 2015-07-31
Lojas: 1115


,Store,DayOfWeek,Date,Sales,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,...,Ano,Mes,Dia,SemanaDoAno,DiaDoAno,SemDataConcorrente,MesesConcorrente,MesesPromo2,EmMesPromo2,LojaComGap
0,1,3,2013-01-02,5530,0,0,1,2,0,1270.0,...,2013,1,2,1,2,0,52.0,0.0,0,0
1,1,4,2013-01-03,4327,0,0,1,2,0,1270.0,...,2013,1,3,1,3,0,52.0,0.0,0,0
2,1,5,2013-01-04,4486,0,0,1,2,0,1270.0,...,2013,1,4,1,4,0,52.0,0.0,0,0
3,1,6,2013-01-05,4997,0,0,1,2,0,1270.0,...,2013,1,5,1,5,0,52.0,0.0,0,0
4,1,1,2013-01-07,7176,1,0,1,2,0,1270.0,...,2013,1,7,2,7,0,52.0,0.0,0,0


### 0.1 O que ja veio pronto do notebook 01

Confere as colunas existentes antes de criar qualquer variavel, para nao gerar
duplicata com nome diferente (por exemplo `Dia` e `DiaDoMes` com o mesmo conteudo).

In [ ]:
print('Colunas disponiveis:')
print(sorted(df.columns.tolist()))
print()

nulos = df.isna().sum()
print('Nulos por coluna:')
print(nulos[nulos > 0] if nulos.sum() else 'nenhum')

Colunas disponiveis:
['Ano', 'Assortment', 'CompetitionDistance', 'Date', 'DayOfWeek', 'Dia', 'DiaDoAno', 'EmMesPromo2', 'LojaComGap', 'Mes', 'MesesConcorrente', 'MesesPromo2', 'Promo', 'Promo2', 'Sales', 'SchoolHoliday', 'SemDataConcorrente', 'SemInfoConcorrente', 'SemanaDoAno', 'StateHoliday', 'Store', 'StoreType']

Nulos por coluna:
nenhum


In [ ]:
# o notebook 01 ja removeu os dias com loja fechada; esta celula so confirma
if 'Open' in df.columns:
    print('Registros com loja fechada:', (df['Open'] == 0).sum())
else:
    print('Coluna Open ausente - filtro ja aplicado no notebook 01')

print('Registros com Sales = 0:', (df['Sales'] == 0).sum())
print('Total de linhas:', len(df))

Coluna Open ausente - filtro ja aplicado no notebook 01
Registros com Sales = 0: 0
Total de linhas: 844338


## 1. Variaveis temporais

O calendario carrega boa parte da sazonalidade do varejo: concentracao de compras
no fim do mes (ciclo de pagamento), movimento diferente por dia da semana e picos
de fim de ano. Modelos de arvore nao enxergam a data como sequencia temporal, entao
cada informacao do calendario precisa estar explicita em uma coluna.

O notebook `01` ja criou a maior parte dessas variaveis. A celula abaixo cria
apenas o que estiver faltando, usando os nomes ja adotados no projeto.

In [ ]:
# cria cada variavel apenas se ela ainda nao existir
if 'Ano' not in df.columns:
    df['Ano'] = df['Date'].dt.year
if 'Mes' not in df.columns:
    df['Mes'] = df['Date'].dt.month
if 'Dia' not in df.columns:
    df['Dia'] = df['Date'].dt.day
if 'DiaDoAno' not in df.columns:
    df['DiaDoAno'] = df['Date'].dt.dayofyear
if 'SemanaDoAno' not in df.columns:
    df['SemanaDoAno'] = df['Date'].dt.isocalendar().week.astype(int)
if 'Trimestre' not in df.columns:
    df['Trimestre'] = df['Date'].dt.quarter

# proxy do ciclo de pagamento
if 'InicioMes' not in df.columns:
    df['InicioMes'] = (df['Dia'] <= 10).astype(int)
if 'FimMes' not in df.columns:
    df['FimMes'] = (df['Dia'] >= 25).astype(int)

temporais = ['Ano', 'Mes', 'Dia', 'DiaDoAno', 'SemanaDoAno',
             'Trimestre', 'DayOfWeek', 'InicioMes', 'FimMes']
df[['Date'] + temporais].head()

,Date,Ano,Mes,Dia,DiaDoAno,SemanaDoAno,Trimestre,DayOfWeek,InicioMes,FimMes
0,2013-01-02,2013,1,2,2,1,1,3,1,0
1,2013-01-03,2013,1,3,3,1,1,4,1,0
2,2013-01-04,2013,1,4,4,1,1,5,1,0
3,2013-01-05,2013,1,5,5,1,1,6,1,0
4,2013-01-07,2013,1,7,7,2,1,1,1,0


## 2. Cadastro da loja

O notebook `01` ja fez o merge com o `store.csv` e ja calculou `MesesConcorrente`,
`MesesPromo2` e `EmMesPromo2`.

**Imputacao da distancia do concorrente (feita no notebook 01).** As lojas sem
`CompetitionDistance` receberam a **mediana** da coluna, acompanhada da flag
`SemInfoConcorrente`, que marca quais registros foram imputados. A flag e o que
torna a imputacao segura: sem ela, o modelo trataria um valor inventado como se
fosse medido; com ela, pode isolar esses casos.

Ressalva metodologica a declarar na monografia: a mediana foi calculada sobre a
base completa, incluindo o periodo de teste. Como a distancia e um atributo fixo
da loja e nao varia no tempo, o efeito e desprezivel - a mediana seria praticamente
a mesma se calculada apenas no treino.

**Escala logaritmica.** Unica transformacao feita aqui. A diferenca entre 100 e 500
metros importa muito mais para a concorrencia do que a diferenca entre 20 e 25
quilometros, e o log reflete isso, alem de reduzir o peso de valores extremos.

In [ ]:
# a imputacao ja foi feita no notebook 01; aqui apenas confirmamos o contrato
assert df['CompetitionDistance'].isna().sum() == 0,     'CompetitionDistance com nulos - conferir o Passo 4 do notebook 01'

df['LogDistConcorrente'] = np.log1p(df['CompetitionDistance'])

print('Registros imputados no notebook 01 (SemInfoConcorrente = 1):',
      df['SemInfoConcorrente'].sum())
print('Lojas afetadas:', df[df['SemInfoConcorrente'] == 1]['Store'].nunique())
print()
print(df[['CompetitionDistance', 'LogDistConcorrente']].describe())

Registros imputados no notebook 01 (SemInfoConcorrente = 1): 2186
Lojas afetadas: 3

       CompetitionDistance  LogDistConcorrente
count        844338.000000       844338.000000
mean           5450.031907            7.644958
std            7801.087197            1.558245
min              20.000000            3.044522
25%             710.000000            6.566672
50%            2320.000000            7.749753
75%            6880.000000            8.836519
max           75860.000000           11.236658


In [ ]:
# conferencia das variaveis de concorrencia e Promo2 vindas do notebook 01
for col in ['MesesConcorrente', 'MesesPromo2', 'EmMesPromo2',
            'SemInfoConcorrente', 'SemDataConcorrente']:
    if col in df.columns:
        print(f'{col}: {df[col].isna().sum()} nulos | '
              f'min {df[col].min()} | max {df[col].max()}')
    else:
        print(f'{col}: AUSENTE - verificar notebook 01')

MesesConcorrente: 0 nulos | min 0.0 | max 1386.0
MesesPromo2: 0 nulos | min 0.0 | max 72.0
EmMesPromo2: 0 nulos | min 0 | max 1
SemInfoConcorrente: 0 nulos | min 0 | max 1
SemDataConcorrente: 0 nulos | min 0 | max 1


## 3. Medias historicas por loja

Este projeto **nao usa lags nem medias moveis** (venda de 7 dias atras, media das
ultimas 4 semanas). Em um horizonte de previsao de varias semanas, esses valores
nao existem no momento em que a previsao e feita, e usa-los produziria um modelo
com desempenho artificial e sem utilidade operacional.

No lugar deles, o patamar de vendas de cada loja entra por **medias historicas**:
um valor unico por loja, calculado uma unica vez sobre o periodo de treino e colado
como coluna fixa. Como nao depende de dado recente, funciona para qualquer horizonte.

**Regra que nao pode ser quebrada:** as medias sao calculadas somente com dados
anteriores a `DATA_CORTE`. Usar a base inteira contaminaria o conjunto de teste.
A mesma data de corte deve ser usada no notebook `03`.

In [ ]:
# data de corte treino/teste - precisa ser a MESMA no notebook 03
DATA_CORTE = pd.Timestamp('2015-06-19')   # ultimas 6 semanas reservadas para teste

treino = df[df['Date'] < DATA_CORTE]
teste  = df[df['Date'] >= DATA_CORTE]

print('Treino:', len(treino), 'linhas |',
      treino['Date'].min().date(), 'a', treino['Date'].max().date())
print('Teste: ', len(teste), 'linhas |',
      teste['Date'].min().date(), 'a', teste['Date'].max().date())

media_geral = treino['Sales'].mean()
print('\nMedia geral de vendas no treino:', round(media_geral, 2))

Treino: 802942 linhas | 2013-01-01 a 2015-06-18
Teste:  41396 linhas | 2015-06-19 a 2015-07-31

Media geral de vendas no treino: 6953.94


In [ ]:
# 1) patamar de vendas da loja
m_loja = treino.groupby('Store')['Sales'].mean().rename('MediaLoja')

# 2) patamar por dia da semana (segunda vende diferente de sabado)
m_loja_dow = (treino.groupby(['Store', 'DayOfWeek'])['Sales']
                    .mean().rename('MediaLojaDiaSemana'))

# 3) patamar com e sem promocao (sensibilidade de cada loja a promocao)
m_loja_promo = (treino.groupby(['Store', 'Promo'])['Sales']
                      .mean().rename('MediaLojaPromo'))

antes = len(df)
df = df.merge(m_loja,       on='Store',                how='left')
df = df.merge(m_loja_dow,   on=['Store', 'DayOfWeek'], how='left')
df = df.merge(m_loja_promo, on=['Store', 'Promo'],     how='left')

assert len(df) == antes, 'Os merges alteraram a contagem de linhas'
print('Linhas preservadas:', len(df))

# combinacoes ausentes no treino recebem a media geral
for col in ['MediaLoja', 'MediaLojaDiaSemana', 'MediaLojaPromo']:
    faltando = df[col].isna().sum()
    if faltando:
        print(f'{col}: {faltando} linhas preenchidas com a media geral')
    df[col] = df[col].fillna(media_geral)

df[['Store', 'Date', 'Sales', 'MediaLoja',
    'MediaLojaDiaSemana', 'MediaLojaPromo']].head(10)

Linhas preservadas: 844338


,Store,Date,Sales,MediaLoja,MediaLojaDiaSemana,MediaLojaPromo
0,1,2013-01-02,5530,4777.599462,4571.230159,4342.426829
1,1,2013-01-03,4327,4777.599462,4462.440678,4342.426829
2,1,2013-01-04,4486,4777.599462,4753.303279,4342.426829
3,1,2013-01-05,4997,4777.599462,4977.695312,4342.426829
4,1,2013-01-07,7176,4777.599462,5195.303279,5311.793413
5,1,2013-01-08,5580,4777.599462,4696.218750,5311.793413
6,1,2013-01-09,5471,4777.599462,4571.230159,5311.793413
7,1,2013-01-10,4892,4777.599462,4462.440678,5311.793413
8,1,2013-01-11,4881,4777.599462,4753.303279,5311.793413
9,1,2013-01-12,4952,4777.599462,4977.695312,4342.426829


### 3.1 Verificacao de vazamento

As medias precisam ser constantes por loja ao longo de todo o periodo, inclusive
no teste. Se variarem, houve recalculo indevido com dados posteriores a data de corte.

In [ ]:
valores_por_loja = df.groupby('Store')['MediaLoja'].nunique()
print('Valores distintos de MediaLoja por loja:')
print(valores_por_loja.value_counts())

assert valores_por_loja.max() == 1, 'VAZAMENTO: MediaLoja varia dentro da mesma loja'
print('\nOK - medias constantes por loja, sem vazamento')

Valores distintos de MediaLoja por loja:
MediaLoja
1    1115
Name: count, dtype: int64

OK - medias constantes por loja, sem vazamento


## 4. Variaveis macroeconomicas (FRED)

Quatro series mensais da economia alema, baixadas do FRED e armazenadas em
`Bases Rossmann/`:

| Codigo FRED | Serie | Frequencia |
|---|---|---|
| `CSCICP02DEM460S` | Indice de confianca do consumidor | Mensal |
| `DEUCPIALLMINMEI` | Indice de precos ao consumidor (inflacao) | Mensal |
| `DEUSARTMISMEI` | Volume de vendas no varejo, com ajuste sazonal | Mensal |
| `LMUNRRTTDEM156S` | Taxa de desemprego | Mensal |

A serie de varejo substitui a `DEUSARTAISMEI` usada inicialmente, que e **anual**
e por isso oferecia apenas tres observacoes no periodo das vendas. A versao mensal
mede **volume** (quantidade real vendida), e nao valor em euros. Isso e desejavel:
as vendas do Rossmann ja estao em euros e portanto embutem inflacao, que entra no
modelo separadamente pelo IPC. Usar volume evita que o mesmo efeito inflacionario
apareca em duas variaveis.

Entre as duas versoes mensais disponiveis, a escolhida tem ajuste sazonal. A versao
sem ajuste (`DEUSLRTTO01IXOBM`) traria de volta a sazonalidade do varejo alemao,
que o modelo ja captura por mes, semana do ano e dia da semana.

Duas decisoes metodologicas nesta secao.

**Defasagem de publicacao.** Um indicador mensal referente a junho nao esta
disponivel em junho: e publicado semanas depois. Fazer o merge direto pelo mes de
referencia colocaria no modelo uma informacao que ele nao teria no momento da
previsao - o mesmo problema das variaveis de lag, em outra forma. Por isso o dado
do mes M e atribuido ao mes M+1 das vendas, controlado por `DEFASAGEM_MESES`.

Ressalva a declarar na monografia: o dado de varejo alemao costuma ser publicado
perto do fim do mes seguinte ao de referencia, entao para essa serie especifica
uma defasagem de dois meses seria mais rigorosa. Optou-se por uma defasagem unica
para todas as series, por simplicidade de manutencao. Testar `DEFASAGEM_MESES = 2`
e uma analise de sensibilidade de baixo custo, sugerida como trabalho futuro.

**Variacao em vez de nivel.** As series sao indices com tendencia: o indice de
precos sobe de forma quase monotonica ao longo do periodo. Modelos de arvore nao
extrapolam alem do intervalo observado no treino, entao um indice sempre crescente
assume no teste valores que o modelo nunca viu, o que torna a variavel inutil ou
enganosa. As series entram como variacao percentual em 12 meses, que e estacionaria
e tem leitura economica direta ("inflacao de 2,1% no ano").

**Variacao percentual ou diferenca absoluta?** Nem toda serie aceita variacao
percentual. O indice de confianca do consumidor e um indicador de saldo: oscila
perto de zero e chega a trocar de sinal. Dividir por uma base proxima de zero
produz variacoes de centenas de pontos percentuais, sem qualquer significado
economico - foi o que gerou os valores infinitos observados na primeira execucao.

Para essa serie usa-se a **diferenca absoluta** em 12 meses ("subiu 0,4 ponto no
ano"), que e como bancos centrais reportam indicadores de sentimento. As outras
tres seguem em variacao percentual. O notebook calcula as duas transformacoes para
todas as series; a escolha e feita na lista de features.

Os niveis originais tambem sao mantidos na base para uso na analise exploratoria
e no dashboard, mas ficam fora da lista de features do modelo.

In [ ]:
CAMINHO_BASES = CAMINHO + 'Bases Rossmann/'

SERIES = {
    'CSCICP02DEM460S': 'ConfiancaConsumidor',
    'DEUCPIALLMINMEI': 'InflacaoIndice',
    'DEUSARTMISMEI':   'VarejoVolume',
    'LMUNRRTTDEM156S': 'Desemprego',
}

# quantos meses de atraso entre o mes de referencia do indicador e sua publicacao
DEFASAGEM_MESES = 1

# True baixa do FRED e salva uma copia local; False le apenas os CSVs da pasta.
# A copia local mantem o notebook reproduzivel mesmo se o FRED estiver indisponivel.
BAIXAR_DO_FRED = True

def carregar_serie(codigo):
    if BAIXAR_DO_FRED:
        url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={codigo}'
        bruta = pd.read_csv(url)
        bruta.to_csv(CAMINHO_BASES + codigo + '.csv', index=False)
        return bruta
    return pd.read_csv(CAMINHO_BASES + codigo + '.csv')

macro = None
for arquivo, nome in SERIES.items():
    serie = carregar_serie(arquivo)

    # o FRED ja usou 'DATE' e hoje usa 'observation_date' - pega pela posicao
    col_data, col_valor = serie.columns[0], serie.columns[1]
    serie = serie.rename(columns={col_data: 'DataMacro', col_valor: nome})

    serie['DataMacro'] = pd.to_datetime(serie['DataMacro'])
    serie[nome] = pd.to_numeric(serie[nome], errors='coerce')
    serie = serie[['DataMacro', nome]].sort_values('DataMacro')

    # duas transformacoes de 12 meses; a escolha por serie e feita na lista de features
    # variacao percentual: adequada a indices que nao se aproximam de zero
    serie[nome + '_var12m'] = (serie[nome]
                               .pct_change(12, fill_method=None)
                               .replace([np.inf, -np.inf], np.nan) * 100)

    # diferenca absoluta: adequada a indices de saldo ou sentimento, que oscilam
    # perto de zero e tornam a variacao percentual instavel
    serie[nome + '_dif12m'] = serie[nome].diff(12)

    macro = serie if macro is None else macro.merge(serie, on='DataMacro', how='outer')
    print(f'{nome}: {len(serie)} observacoes | '
          f'{serie["DataMacro"].min().date()} a {serie["DataMacro"].max().date()}')

macro = macro.sort_values('DataMacro').reset_index(drop=True)
assert macro['DataMacro'].is_unique, 'Datas repetidas na tabela macro'
print('\nTabela macro consolidada:', macro.shape)
macro.tail()

ConfiancaConsumidor: 642 observacoes | 1973-01-01 a 2026-06-01
InflacaoIndice: 843 observacoes | 1955-01-01 a 2025-03-01
VarejoVolume: 831 observacoes | 1955-01-01 a 2024-03-01
Desemprego: 661 observacoes | 1969-01-01 a 2024-01-01

Tabela macro consolidada: (858, 13)


,DataMacro,ConfiancaConsumidor,ConfiancaConsumidor_var12m,ConfiancaConsumidor_dif12m,InflacaoIndice,InflacaoIndice_var12m,InflacaoIndice_dif12m,VarejoVolume,VarejoVolume_var12m,VarejoVolume_dif12m,Desemprego,Desemprego_var12m,Desemprego_dif12m
853,2026-02-01,-10.8,-4.424779,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
854,2026-03-01,-13.6,33.333333,-3.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
855,2026-04-01,-17.8,67.924528,-7.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
856,2026-05-01,-16.0,73.913043,-6.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
857,2026-06-01,-14.6,52.083333,-5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 4.1 Diagnostico das series

Confere frequencia, periodo coberto e inicio da variacao em 12 meses. Uma serie
anual ou truncada aparece aqui, antes de contaminar o resto do notebook.

O periodo coberto e reportado para a serie inteira, mas o alerta sobre proximidade
de zero avalia apenas a janela usada pelo modelo - o periodo das vendas mais os 12
meses anteriores, necessarios para a variacao anual. Sem esse recorte, valores de
decadas atras disparariam alarme sobre dados que o modelo nunca vera.

In [ ]:
print('Periodo das vendas:', df['Date'].min().date(), 'a', df['Date'].max().date())
print()

for nome in SERIES.values():
    nivel = macro.loc[macro[nome].notna(), 'DataMacro']
    variacao = macro.loc[macro[nome + '_var12m'].notna(), 'DataMacro']
    intervalo = nivel.diff().dt.days.median()
    freq = ('mensal' if intervalo < 40 else
            'trimestral' if intervalo < 100 else 'ANUAL - INADEQUADA')

    print(f'{nome}  ({len(nivel)} observacoes, {freq})')
    print(f'  nivel:    {nivel.min().date()} a {nivel.max().date()}')
    if len(variacao):
        print(f'  var 12m:  {variacao.min().date()} a {variacao.max().date()}')
    else:
        print('  var 12m:  VAZIA - serie curta demais para variacao anual')

    if nivel.max() < df['Date'].max():
        atraso = (df['Date'].max() - nivel.max()).days
        print(f'  ATENCAO: termina {atraso} dias antes do fim das vendas')

    # series que se aproximam de zero nao aceitam variacao percentual.
    # verifica apenas o periodo efetivamente usado, nao a serie historica inteira:
    # valores de decadas atras podem disparar alarme falso sobre dados que o
    # modelo nunca vera (o desemprego alemao ficou abaixo de 1% nos anos 1970).
    inicio = df['Date'].min() - pd.DateOffset(months=12)
    valores = macro.loc[macro['DataMacro'].between(inicio, df['Date'].max()), nome].dropna()

    if len(valores) and (valores.abs().min() < 1 or (valores.min() < 0 < valores.max())):
        print('  ATENCAO: serie passa perto de zero no periodo usado'
              ' - usar diferenca absoluta (_dif12m)')
    print()

Periodo das vendas: 2013-01-01 a 2015-07-31

ConfiancaConsumidor  (642 observacoes, mensal)
  nivel:    1973-01-01 a 2026-06-01
  var 12m:  1974-01-01 a 2026-06-01
  ATENCAO: serie passa perto de zero no periodo usado - usar diferenca absoluta (_dif12m)

InflacaoIndice  (843 observacoes, mensal)
  nivel:    1955-01-01 a 2025-03-01
  var 12m:  1956-01-01 a 2025-03-01

VarejoVolume  (831 observacoes, mensal)
  nivel:    1955-01-01 a 2024-03-01
  var 12m:  1956-01-01 a 2024-03-01

Desemprego  (661 observacoes, mensal)
  nivel:    1969-01-01 a 2024-01-01
  var 12m:  1970-01-01 a 2024-01-01



In [ ]:
# aplica a defasagem: o indicador do mes M passa a valer para o mes M + DEFASAGEM_MESES
macro['DataReferencia'] = macro['DataMacro'] + pd.DateOffset(months=DEFASAGEM_MESES)
macro['Ano'] = macro['DataReferencia'].dt.year
macro['Mes'] = macro['DataReferencia'].dt.month

colunas_macro = [c for c in macro.columns
                 if c not in ['DataMacro', 'DataReferencia', 'Ano', 'Mes']]

antes = len(df)
df = df.merge(macro[['Ano', 'Mes'] + colunas_macro], on=['Ano', 'Mes'], how='left')

assert len(df) == antes, 'O merge macro alterou a contagem de linhas'
print('Linhas preservadas:', len(df))

print('\nCobertura (nulos por variavel macro):')
print(df[colunas_macro].isna().sum())

Linhas preservadas: 844338

Cobertura (nulos por variavel macro):
ConfiancaConsumidor               0
ConfiancaConsumidor_var12m    29079
ConfiancaConsumidor_dif12m        0
InflacaoIndice                    0
InflacaoIndice_var12m             0
InflacaoIndice_dif12m             0
VarejoVolume                      0
VarejoVolume_var12m               0
VarejoVolume_dif12m               0
Desemprego                        0
Desemprego_var12m                 0
Desemprego_dif12m                 0
dtype: int64


### 4.2 Preenchimento: apenas para frente

Meses de venda sem indicador correspondente recebem o **ultimo valor conhecido**
(`ffill`). E o que estaria disponivel na pratica no momento da previsao.

O preenchimento para tras (`bfill`) nao e usado, ainda que resolvesse mais casos:
ele traria um valor futuro para preencher o passado, exatamente o vazamento que
motivou a exclusao das variaveis de lag.

Variaveis que continuarem com nulos depois do `ffill` nao podem entrar no modelo,
e sao removidas automaticamente da lista de features.

In [ ]:
antes_fill = df[colunas_macro].isna().sum()

df = df.sort_values(['Store', 'Date'])
df[colunas_macro] = df.groupby('Store')[colunas_macro].ffill()
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

depois_fill = df[colunas_macro].isna().sum()

print('Nulos antes do ffill:')
print(antes_fill[antes_fill > 0] if antes_fill.sum() else 'nenhum')
print()
print('Nulos apos o ffill:')
print(depois_fill[depois_fill > 0] if depois_fill.sum() else 'nenhum')

# guarda a lista para excluir da tabela final
macro_inutilizaveis = depois_fill[depois_fill > 0].index.tolist()
if macro_inutilizaveis:
    print('\nVariaveis macro inutilizaveis (serao removidas):', macro_inutilizaveis)

Nulos antes do ffill:
ConfiancaConsumidor_var12m    29079
dtype: int64

Nulos apos o ffill:
nenhum


In [ ]:
# amostra: um valor por mes, para conferir visualmente a defasagem
(df[['Date', 'Ano', 'Mes'] + colunas_macro]
   .drop_duplicates(['Ano', 'Mes'])
   .tail(8))

,Date,Ano,Mes,ConfiancaConsumidor,ConfiancaConsumidor_var12m,ConfiancaConsumidor_dif12m,InflacaoIndice,InflacaoIndice_var12m,InflacaoIndice_dif12m,VarejoVolume,VarejoVolume_var12m,VarejoVolume_dif12m,Desemprego,Desemprego_var12m,Desemprego_dif12m
581,2014-12-01,2014,12,-2.0,11.111111,-0.2,99.54263,0.565502,0.55975,96.70806,0.311198,0.30002,6.6,-4.347826,-0.3
606,2015-01-02,2015,1,-1.3,0.000000,0.0,99.54263,0.187789,0.18658,98.10818,2.400844,2.30020,6.5,-4.411765,-0.3
632,2015-02-02,2015,2,-0.5,66.666667,-0.2,98.51642,-0.283290,-0.27988,98.50821,2.925806,2.80023,6.5,-4.411765,-0.3
656,2015-03-02,2015,3,-0.2,66.666667,-0.2,99.21654,-0.046563,-0.04622,98.60822,2.388368,2.30019,6.5,-4.411765,-0.3
682,2015-04-01,2015,4,0.5,-183.333333,1.1,99.71662,0.174789,0.17399,98.30819,1.549582,1.50012,6.5,-4.411765,-0.3
706,2015-05-02,2015,5,0.6,-250.000000,1.0,100.21670,0.866228,0.86065,99.00825,2.803733,2.70022,6.5,-2.985075,-0.2
729,2015-06-01,2015,6,0.2,-300.000000,0.3,100.41670,1.162510,1.15394,100.20840,4.921516,4.70044,6.4,-4.477612,-0.3
754,2015-07-01,2015,7,-0.3,-25.000000,0.1,100.41670,0.878086,0.87407,99.50829,2.577311,2.50020,6.4,-4.477612,-0.3


### 4.3 Limitacao a declarar na monografia

O conjunto de teste cobre seis semanas, ou seja, um a dois meses de calendario.
Nesse intervalo as variaveis macroeconomicas sao praticamente constantes, e por
isso **nao e possivel validar o poder preditivo delas com este desenho de teste**.

A contribuicao dessas variaveis ao trabalho e outra: sustentar a analise de
cenario economico e alimentar as respostas do assistente do dashboard sobre o
contexto em que as vendas ocorreram. E uma escolha consciente, e deve constar
como tal na secao de Resultados.

## 5. Verificacao da codificacao herdada do notebook 01

A codificacao das categoricas ja foi feita no **Passo 10 do notebook 01**, que
converteu `StoreType`, `Assortment` e `StateHoliday` em numeros no lugar, usando
codificacao ordinal simples: cada letra vira um numero.

O criterio: modelos de arvore nao interpretam esses numeros como ordem de grandeza -
eles apenas separam os grupos. Por isso a codificacao ordinal basta, sem necessidade
de one-hot encoding, que multiplicaria o numero de colunas sem ganho.

Os mapas usados no notebook 01 foram `a`=0, `b`=1, `c`=2, `d`=3 para `StoreType`;
`a`=0, `b`=1, `c`=2 para `Assortment`; e `0`=0, `a`=1, `b`=2, `c`=3 para
`StateHoliday`, com o cuidado de tratar o valor `0` que aparece ora como texto,
ora como numero na base original.

Esta secao apenas confirma que a conversao chegou intacta.

In [ ]:
categoricas = ['StoreType', 'Assortment', 'StateHoliday']

for col in categoricas:
    print(f'{col} ({df[col].dtype}): valores {sorted(df[col].unique())}')

# se alguma chegar como texto, o notebook 01 nao rodou por completo
nao_convertidas = [c for c in categoricas if not pd.api.types.is_numeric_dtype(df[c])]
assert not nao_convertidas,     f'Categoricas ainda em texto: {nao_convertidas} - reexecutar o Passo 10 do notebook 01'

print()
print('OK - todas convertidas no notebook 01, sem nulos:',
      df[categoricas].isna().sum().sum() == 0)

StoreType (int64): valores [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Assortment (int64): valores [np.int64(0), np.int64(1), np.int64(2)]
StateHoliday (int64): valores [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

OK - todas convertidas no notebook 01, sem nulos: True


## 6. Tabela final de modelagem

Seleciono apenas as colunas que entram no modelo, mais `Date` e `Sales`, necessarias
para a divisao treino/teste por tempo no notebook `03`.

`SemInfoConcorrente` e `SemDataConcorrente` sao mantidas: marcam ausencia de cadastro,
e a propria ausencia costuma ser informativa para o modelo.

In [ ]:
features = [
    # temporais
    'Ano', 'Mes', 'Dia', 'DiaDoAno', 'SemanaDoAno', 'Trimestre',
    'DayOfWeek', 'InicioMes', 'FimMes',
    # promocoes e feriados
    'Promo', 'Promo2', 'EmMesPromo2', 'MesesPromo2',
    'StateHoliday', 'SchoolHoliday',
    # cadastro da loja
    'Store', 'StoreType', 'Assortment',
    'LogDistConcorrente', 'MesesConcorrente',
    'SemInfoConcorrente', 'SemDataConcorrente', 'LojaComGap',
    # macroeconomicas (transformacoes de 12 meses, defasadas)
    # confianca do consumidor em diferenca absoluta; as demais em variacao percentual
    'ConfiancaConsumidor_dif12m', 'InflacaoIndice_var12m',
    'VarejoVolume_var12m', 'Desemprego_var12m',
    # medias historicas (calculadas somente no treino)
    'MediaLoja', 'MediaLojaDiaSemana', 'MediaLojaPromo',
]

ausentes = [c for c in features if c not in df.columns]
if ausentes:
    print('ATENCAO - features ausentes na base, serao ignoradas:', ausentes)
    features = [c for c in features if c in df.columns]

# remove variaveis macro que ficaram com nulos apos o ffill
descartadas = [c for c in features if c in macro_inutilizaveis]
if descartadas:
    print('ATENCAO - features macro descartadas por nulos:', descartadas)
    features = [c for c in features if c not in descartadas]

base_modelagem = df[['Date', 'Sales'] + features].copy()

print('Formato final:', base_modelagem.shape)
print('Total de features:', len(features))
print('Nulos restantes:', base_modelagem.isna().sum().sum())
base_modelagem.head()

Formato final: (844338, 32)
Total de features: 30
Nulos restantes: 0


,Date,Sales,Ano,Mes,Dia,DiaDoAno,SemanaDoAno,Trimestre,DayOfWeek,InicioMes,...,SemInfoConcorrente,SemDataConcorrente,LojaComGap,ConfiancaConsumidor_dif12m,InflacaoIndice_var12m,VarejoVolume_var12m,Desemprego_var12m,MediaLoja,MediaLojaDiaSemana,MediaLojaPromo
0,2013-01-02,5530,2013,1,2,2,1,1,3,1,...,0,0,0,-2.9,2.040813,0.105487,1.470588,4777.599462,4571.230159,4342.426829
1,2013-01-03,4327,2013,1,3,3,1,1,4,1,...,0,0,0,-2.9,2.040813,0.105487,1.470588,4777.599462,4462.440678,4342.426829
2,2013-01-04,4486,2013,1,4,4,1,1,5,1,...,0,0,0,-2.9,2.040813,0.105487,1.470588,4777.599462,4753.303279,4342.426829
3,2013-01-05,4997,2013,1,5,5,1,1,6,1,...,0,0,0,-2.9,2.040813,0.105487,1.470588,4777.599462,4977.695312,4342.426829
4,2013-01-07,7176,2013,1,7,7,2,1,1,1,...,0,0,0,-2.9,2.040813,0.105487,1.470588,4777.599462,5195.303279,5311.793413


In [ ]:
base_modelagem.to_parquet(CAMINHO + 'base_modelagem_02.parquet', index=False)
print('Salvo em', CAMINHO + 'base_modelagem_02.parquet')

# guarda a data de corte para reuso no notebook 03
with open(CAMINHO + 'data_corte.txt', 'w') as f:
    f.write(str(DATA_CORTE.date()))
print('Data de corte registrada:', DATA_CORTE.date())

# registra a lista de features usada, para constar na monografia
with open(CAMINHO + 'features_02.txt', 'w') as f:
    f.write('\n'.join(features))
print('Lista de features registrada:', len(features), 'variaveis')

Salvo em /content/drive/MyDrive/Trabalho MBA/base_modelagem_02.parquet
Data de corte registrada: 2015-06-19
Lista de features registrada: 30 variaveis


## 7. Conferencias finais

Seis verificacoes antes de seguir para o notebook `03`. As primeiras procuram
problemas que passam despercebidos no merge e so apareceriam como resultado
estranho na modelagem. A 7.4 estabelece a referencia de comparacao do projeto.

### 7.1 Integridade da tabela

Valores infinitos nao aparecem em `isna()` mas quebram o XGBoost. Duplicatas de
loja/dia indicariam merge mal feito. E `Sales` na lista de features seria vazamento
direto do alvo.

In [ ]:
print('Formato:', base_modelagem.shape)
print('Features:', len(features))
print('Nulos:', base_modelagem.isna().sum().sum())

numericas = base_modelagem.select_dtypes(include=[np.number])
print('Infinitos:', np.isinf(numericas).sum().sum())

print('Duplicatas Store+Date:', base_modelagem.duplicated(['Store', 'Date']).sum())
print('Vazamento do alvo (Sales nas features):', 'Sales' in features)

Formato: (844338, 32)
Features: 30
Nulos: 0
Infinitos: 0
Duplicatas Store+Date: 0
Vazamento do alvo (Sales nas features): False


### 7.2 Plausibilidade das variaveis macro

Um erro de coluna ou de escala no merge passa silencioso. A inflacao alema entre
2013 e 2015 rodava perto de zero a 2% ao ano, e o desemprego caia lentamente.
Valores na casa das dezenas, ou proximos de zero em excesso, indicam escala trocada.

In [ ]:
colunas_var = [c for c in features if 'var12m' in c]
print(base_modelagem[colunas_var].describe().round(2))

       InflacaoIndice_var12m  VarejoVolume_var12m  Desemprego_var12m
count              844338.00            844338.00          844338.00
mean                    1.10                 1.22              -1.45
std                     0.55                 1.23               2.19
min                    -0.28                -1.04              -4.48
25%                     0.85                 0.32              -4.29
50%                     1.24                 0.95              -1.45
75%                     1.44                 2.01               0.00
max                     2.04                 4.92               1.47


### 7.3 Divisao treino/teste

Lojas presentes apenas no teste nao teriam media historica real - receberiam a
media geral, o que reduziria a qualidade da previsao para elas.

In [ ]:
treino_f = base_modelagem[base_modelagem['Date'] < DATA_CORTE]
teste_f  = base_modelagem[base_modelagem['Date'] >= DATA_CORTE]

print('Treino:', len(treino_f), '|', treino_f['Store'].nunique(), 'lojas')
print('Teste: ', len(teste_f), '|', teste_f['Store'].nunique(), 'lojas')
print('Proporcao do teste:', round(len(teste_f) / len(base_modelagem) * 100, 1), '%')

novas_lojas = set(teste_f['Store']) - set(treino_f['Store'])
print('Lojas presentes so no teste:', len(novas_lojas))

Treino: 802942 | 1115 lojas
Teste:  41396 | 1115 lojas
Proporcao do teste: 4.9 %
Lojas presentes so no teste: 0


### 7.4 Baseline ingenuo

Erro de prever simplesmente a media historica da loja no dia da semana, sem
modelo nenhum. Serve a dois propositos.

No notebook `03`, se o XGBoost nao superar esses numeros com folga, o problema
esta na modelagem e nao nos dados - e isso se descobre em minutos.

Na monografia, apresentar o ganho sobre um baseline explicito e muito mais
informativo que um RMSE isolado, cuja qualidade o leitor nao tem como julgar.

**As quatro metricas.** RMSE e MAE medem erro em euros; o RMSE pune mais os erros
grandes, por elevar ao quadrado. MAPE e RMSPE medem erro relativo, em percentual -
util porque um erro de 500 euros pesa de forma diferente numa loja que vende 3.000
e em outra que vende 15.000.

O **RMSPE** e a metrica oficial da competicao Rossmann no Kaggle, e por isso vale
calcula-lo mesmo sem submeter: permite comparar o resultado deste trabalho com a
literatura da competicao. A regra oficial ignora dias com venda zero; aqui isso nao
tem efeito pratico, porque o notebook `01` ja removeu esses registros, mas a mascara
fica na funcao para manter a implementacao fiel a definicao.

Referencia de calibragem: as solucoes vencedoras da competicao ficaram proximas de
10% de RMSPE, com feature engineering pesada baseada em lags. Sem lags - decisao
documentada na secao 3 - algo em torno de 15% seria um bom resultado.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


def rmspe(y_real, y_previsto):
    """Root Mean Square Percentage Error - metrica oficial da competicao Rossmann.
    Dias com venda zero sao ignorados, conforme a regra do Kaggle."""
    y_real = np.asarray(y_real, dtype=float)
    y_previsto = np.asarray(y_previsto, dtype=float)

    mascara = y_real != 0
    erro_relativo = (y_real[mascara] - y_previsto[mascara]) / y_real[mascara]

    return np.sqrt(np.mean(erro_relativo ** 2)) * 100


y = teste_f['Sales']
pred_baseline = teste_f['MediaLojaDiaSemana']

rmse  = np.sqrt(mean_squared_error(y, pred_baseline))
mae   = mean_absolute_error(y, pred_baseline)
mape  = (abs(y - pred_baseline) / y).mean() * 100
rmspe_valor = rmspe(y, pred_baseline)

print('BASELINE - media historica da loja por dia da semana')
print(f'  RMSE:  {rmse:,.0f}')
print(f'  MAE:   {mae:,.0f}')
print(f'  MAPE:  {mape:.2f}%')
print(f'  RMSPE: {rmspe_valor:.2f}%   <- metrica oficial da competicao')

# registra para reuso no notebook 03 e na monografia
baseline = pd.DataFrame([{'modelo': 'baseline_media_historica',
                          'rmse': rmse, 'mae': mae,
                          'mape': mape, 'rmspe': rmspe_valor}])
baseline.to_csv(CAMINHO + 'baseline_02.csv', index=False)
print('\nSalvo em', CAMINHO + 'baseline_02.csv')

BASELINE - media historica da loja por dia da semana
  RMSE:  1,650
  MAE:   1,240
  MAPE:  18.27%
  RMSPE: 23.32%   <- metrica oficial da competicao

Salvo em /content/drive/MyDrive/Trabalho MBA/baseline_02.csv


### 7.5 Distribuicao de vendas entre treino e teste

Se as seis semanas de teste caissem num periodo atipico, o modelo seria julgado
por uma janela que nao representa o restante da serie. As medidas centrais - media,
mediana e quartis - sao o que importa aqui; minimos e maximos flutuam apenas porque
o treino tem vinte vezes mais observacoes e portanto mais chance de capturar casos raros.

In [ ]:
print('Vendas medias - treino:', round(treino_f['Sales'].mean()))
print('Vendas medias - teste: ', round(teste_f['Sales'].mean()))
diferenca = (teste_f['Sales'].mean() / treino_f['Sales'].mean() - 1) * 100
print(f'Diferenca: {diferenca:.1f}%')
print()
print('Treino:')
print(treino_f['Sales'].describe().round(0))
print('\nTeste:')
print(teste_f['Sales'].describe().round(0))

print('\nRegistros abaixo de 500 (incidentes operacionais isolados):')
print('  treino:', (treino_f['Sales'] < 500).sum())
print('  teste: ', (teste_f['Sales'] < 500).sum())

Vendas medias - treino: 6954
Vendas medias - teste:  6995
Diferenca: 0.6%

Treino:
count    802942.0
mean       6954.0
std        3107.0
min          46.0
25%        4855.0
50%        6367.0
75%        8359.0
max       38722.0
Name: Sales, dtype: float64

Teste:
count    41396.0
mean      6995.0
std       3044.0
min        569.0
25%       4943.0
50%       6414.0
75%       8383.0
max      41551.0
Name: Sales, dtype: float64

Registros abaixo de 500 (incidentes operacionais isolados):
  treino: 7
  teste:  0


### 7.6 Variaveis constantes

Uma feature com um unico valor nao informa nada ao modelo e ainda entra na contagem
reportada na monografia. Na listagem de cardinalidade, valores baixos sao esperados
nas flags binarias; uma variavel que deveria variar bastante aparecendo com poucos
valores indicaria erro de calculo.

In [ ]:
constantes = [c for c in features if base_modelagem[c].nunique() <= 1]
print('Constantes:', constantes if constantes else 'nenhuma')
print()
print('Valores distintos por feature:')
print(base_modelagem[features].nunique().sort_values())

Constantes: nenhuma

Valores distintos por feature:
InicioMes                        2
Promo2                           2
Promo                            2
SchoolHoliday                    2
EmMesPromo2                      2
FimMes                           2
LojaComGap                       2
SemDataConcorrente               2
SemInfoConcorrente               2
Assortment                       3
Ano                              3
StoreType                        4
StateHoliday                     4
Trimestre                        4
DayOfWeek                        7
Mes                             12
Desemprego_var12m               12
InflacaoIndice_var12m           28
ConfiancaConsumidor_dif12m      28
VarejoVolume_var12m             30
Dia                             31
SemanaDoAno                     52
MesesPromo2                    289
MesesConcorrente               336
DiaDoAno                       365
LogDistConcorrente             654
Store                         1115
Med

---

**Proximo passo:** notebook `03 - Modelagem`, usando `base_modelagem_02.parquet`,
a mesma `DATA_CORTE` para a divisao treino/teste e o `baseline_02.csv` como
referencia minima de desempenho.

**Pendencias:**

- Definicao formal do horizonte de previsao, para declarar na monografia.
- Apos o notebook `04`, comparar o erro do modelo entre lojas com e sem `LojaComGap`.
  Se houver diferenca sistematica, avaliar a criacao de uma variavel marcando os
  dias posteriores a reabertura das lojas reformadas.